# Proyek Pengembangan Machine Learning Pipeline

Notebook ini membangun pipeline TFX untuk klasifikasi kualitas red wine.
Target `good_quality` bernilai 1 jika skor kualitas asli minimal 6.

## 1. Persiapan

Import library TFX, TensorFlow, dan TensorFlow Model Analysis yang dipakai
untuk setiap komponen pipeline.

In [1]:
from pathlib import Path

import pandas as pd
import tensorflow as tf
import tensorflow_model_analysis as tfma
import tfx

from tfx.components import (
    CsvExampleGen,
    Evaluator,
    ExampleValidator,
    Pusher,
    SchemaGen,
    StatisticsGen,
    Trainer,
    Transform,
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import (
    LatestBlessedModelStrategy,
)
from tfx.orchestration.experimental.interactive.interactive_context import (
    InteractiveContext,
)
from tfx.proto import example_gen_pb2, pusher_pb2, trainer_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

print("TensorFlow:", tf.__version__)
print("TFX:", tfx.__version__)

2026-05-24 00:32:37.179488: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-24 00:32:37.325852: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-24 00:32:37.325923: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-24 00:32:37.328827: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-24 00:32:37.344470: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-24 00:32:37.346465: I tensorflow/core/platform/cpu_feature_guard.cc:1

/home/admin/.venvs/dicoding-mlops-aqilaziz/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


2026-05-24 00:32:40.642714: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


/home/admin/.venvs/dicoding-mlops-aqilaziz/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.bigquery_storage_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.bigquery_storage_v1 past that date.
  warnings.warn(message, FutureWarning)


/home/admin/.venvs/dicoding-mlops-aqilaziz/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.pubsub_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.pubsub_v1 past that date.
  warnings.warn(message, FutureWarning)


TensorFlow: 2.15.1
TFX: 1.15.1


In [2]:
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"
PIPELINE_ROOT = PROJECT_ROOT / "aqilaziz-pipeline"
MODULE_ROOT = PROJECT_ROOT / "modules"
SERVING_MODEL_DIR = PIPELINE_ROOT / "serving_model"

TRANSFORM_MODULE = MODULE_ROOT / "wine_transform.py"
TRAINER_MODULE = MODULE_ROOT / "wine_trainer.py"

print("Data root:", DATA_ROOT)
print("Pipeline root:", PIPELINE_ROOT)

Data root: /mnt/d/App/PR/dicoding-mlops-aqilaziz/data
Pipeline root: /mnt/d/App/PR/dicoding-mlops-aqilaziz/aqilaziz-pipeline


## 2. Dataset

Dataset Wine Quality Red berisi atribut kimia red wine. Label biner
`good_quality` dibuat dari kolom kualitas asli.

In [3]:
dataset = pd.read_csv(DATA_ROOT / "winequality-red.csv")
display(dataset.head())
display(dataset["good_quality"].value_counts().rename("count"))

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol,good_quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,0


1    855
0    744
Name: count, dtype: int64

## 3. InteractiveContext

Semua komponen pipeline dijalankan dengan `InteractiveContext` dan
artifact disimpan di folder `aqilaziz-pipeline`.

In [4]:
context = InteractiveContext(pipeline_root=str(PIPELINE_ROOT))

## 4. ExampleGen, StatisticsGen, SchemaGen, dan ExampleValidator

`CsvExampleGen` membaca data CSV dan membaginya menjadi train/eval.
Komponen validasi data membuat statistik, schema, dan memeriksa anomali.

In [5]:
output = example_gen_pb2.Output(
    split_config=example_gen_pb2.SplitConfig(
        splits=[
            example_gen_pb2.SplitConfig.Split(name="train", hash_buckets=8),
            example_gen_pb2.SplitConfig.Split(name="eval", hash_buckets=2),
        ]
    )
)

example_gen = CsvExampleGen(
    input_base=str(DATA_ROOT),
    output_config=output,
)
context.run(example_gen)

statistics_gen = StatisticsGen(examples=example_gen.outputs["examples"])
context.run(statistics_gen)

schema_gen = SchemaGen(
    statistics=statistics_gen.outputs["statistics"],
    infer_feature_shape=True,
)
context.run(schema_gen)

example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"],
    schema=schema_gen.outputs["schema"],
)
context.run(example_validator)

ExecutionResult(
    component_id: ExampleValidator
    execution_id: 4
    outputs:
        anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

## 5. Transform

Komponen `Transform` memakai `modules/wine_transform.py` untuk
menstandarkan fitur numerik dengan z-score.

In [6]:
transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs["schema"],
    module_file=str(TRANSFORM_MODULE),
)
context.run(transform)

running bdist_wheel


running build
running build_py
creating build/lib
copying wine_trainer.py -> build/lib
copying wine_transform.py -> build/lib
installing to /tmp/tmpmp037ayd
running install


/home/admin/.venvs/dicoding-mlops-aqilaziz/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        This deprecation is overdue, please update your project and remove deprecated
        calls to avoid build errors in the future.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


running install_lib
copying build/lib/wine_transform.py -> /tmp/tmpmp037ayd/.
copying build/lib/wine_trainer.py -> /tmp/tmpmp037ayd/.
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'


reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
Copying tfx_user_code_Transform.egg-info to /tmp/tmpmp037ayd/./tfx_user_code_Transform-0.0+8529956fc4e9b299ea2f01b28ffd78c87bb712ae3ec53a3c9123775b82ce542b-py3.10.egg-info
running install_scripts
creating /tmp/tmpmp037ayd/tfx_user_code_transform-0.0+8529956fc4e9b299ea2f01b28ffd78c87bb712ae3ec53a3c9123775b82ce542b.dist-info/WHEEL
creating '/tmp/tmp1585fvqu/tfx_user_code_transform-0.0+8529956fc4e9b299ea2f01b28ffd78c87bb712ae3ec53a3c9123775b82ce542b-py3-none-any.whl' and adding '/tmp/tmpmp037ayd' to it
adding 'wine_trainer.py'
adding 'wine_transform.py'
adding 'tfx_user_code_transform-0.0+8529956fc4e9b299ea2f01b28ffd78c87bb712ae3ec53a3c9123775b82ce542b.dist-info/METADATA'
adding 'tfx_user_code_transform-0.0+8529956fc4e9b299ea2f01b28ffd78c87bb712ae3ec53a3c9123775b82ce542b.dist-info/WHEEL'
adding 'tfx_user_code_transform-0.0+8529956fc4e9b299ea2f01b28ffd78

...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 5
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

## 6. Trainer

Komponen `Trainer` memakai `modules/wine_trainer.py` untuk membuat dan
melatih model Keras.

In [7]:
trainer = Trainer(
    module_file=str(TRAINER_MODULE),
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    train_args=trainer_pb2.TrainArgs(num_steps=120),
    eval_args=trainer_pb2.EvalArgs(num_steps=40),
)
context.run(trainer)

running bdist_wheel


running build
running build_py
creating build/lib
copying wine_trainer.py -> build/lib
copying wine_transform.py -> build/lib


/home/admin/.venvs/dicoding-mlops-aqilaziz/lib/python3.10/site-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        This deprecation is overdue, please update your project and remove deprecated
        calls to avoid build errors in the future.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


installing to /tmp/tmpf1yys3ur
running install
running install_lib
copying build/lib/wine_transform.py -> /tmp/tmpf1yys3ur/.
copying build/lib/wine_trainer.py -> /tmp/tmpf1yys3ur/.
running install_egg_info


running egg_info
creating tfx_user_code_Trainer.egg-info
writing tfx_user_code_Trainer.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Trainer.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Trainer.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
Copying tfx_user_code_Trainer.egg-info to /tmp/tmpf1yys3ur/./tfx_user_code_Trainer-0.0+8529956fc4e9b299ea2f01b28ffd78c87bb712ae3ec53a3c9123775b82ce542b-py3.10.egg-info
running install_scripts
creating /tmp/tmpf1yys3ur/tfx_user_code_trainer-0.0+8529956fc4e9b299ea2f01b28ffd78c87bb712ae3ec53a3c9123775b82ce542b.dist-info/WHEEL
creating '/tmp/tmp1hukczse/tfx_user_code_trainer-0.0+8529956fc4e9b299ea2f01b28ffd78c87bb712ae3ec53a3c9123775b82ce542b-py3-none-any.whl' and adding '/tmp/tmpf1yys3ur' to it
adding 'wine_trainer.py'
adding 'wine_transfo

...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: /mnt/d/App/PR/dicoding-mlops-aqilaziz/aqilaziz-pipeline/Trainer/model/6/Format-Serving/assets


INFO:tensorflow:Assets written to: /mnt/d/App/PR/dicoding-mlops-aqilaziz/aqilaziz-pipeline/Trainer/model/6/Format-Serving/assets


ExecutionResult(
    component_id: Trainer
    execution_id: 6
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

## 7. Resolver dan Evaluator

`Resolver` mengambil baseline blessed model jika tersedia. `Evaluator`
menghitung Binary Accuracy dan AUC, lalu menentukan apakah model layak
di-push.

In [8]:
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing),
).with_id("latest_blessed_model_resolver")
context.run(model_resolver)

eval_config = tfma.EvalConfig(
    model_specs=[
        tfma.ModelSpec(
            label_key="good_quality",
            signature_name="serving_default",
            prediction_key="outputs",
        )
    ],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(
                    class_name="BinaryAccuracy",
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={"value": 0.65}
                        )
                    ),
                ),
                tfma.MetricConfig(
                    class_name="AUC",
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={"value": 0.70}
                        )
                    ),
                ),
            ]
        )
    ],
)

evaluator = Evaluator(
    examples=example_gen.outputs["examples"],
    model=trainer.outputs["model"],
    baseline_model=model_resolver.outputs["model"],
    eval_config=eval_config,
)
context.run(evaluator)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


ExecutionResult(
    component_id: Evaluator
    execution_id: 8
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

## 8. Pusher

Model yang mendapatkan blessing dari Evaluator disimpan sebagai serving
model di dalam folder pipeline.

In [9]:
pusher = Pusher(
    model=trainer.outputs["model"],
    model_blessing=evaluator.outputs["blessing"],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=str(SERVING_MODEL_DIR)
        )
    ),
)
context.run(pusher)

print("Pushed model artifact:")
for artifact in pusher.outputs["pushed_model"].get():
    print(artifact.uri)

Pushed model artifact:
/mnt/d/App/PR/dicoding-mlops-aqilaziz/aqilaziz-pipeline/Pusher/pushed_model/9


## 9. Ringkasan

Seluruh komponen wajib telah dijalankan dengan `InteractiveContext` dan
artifact pipeline tersimpan pada folder `aqilaziz-pipeline`.

In [10]:
print("Pipeline artifact root:", PIPELINE_ROOT)
print("Serving model root:", SERVING_MODEL_DIR)
print("Top-level pipeline artifacts:")
for path in sorted(PIPELINE_ROOT.iterdir()):
    print("-", path.name)

Pipeline artifact root: /mnt/d/App/PR/dicoding-mlops-aqilaziz/aqilaziz-pipeline
Serving model root: /mnt/d/App/PR/dicoding-mlops-aqilaziz/aqilaziz-pipeline/serving_model
Top-level pipeline artifacts:
- CsvExampleGen
- Evaluator
- ExampleValidator
- Pusher
- SchemaGen
- StatisticsGen
- Trainer
- Transform
- _wheels
- metadata.sqlite
- serving_model
